# 07 HPC execution

This notebook is only for execution planning, submission, restart continuation, benchmarking, and performance tuning. It does not generate topology, solvate systems, or modify molecular inputs.


## Workflow Position

```mermaid
flowchart LR
    A5["05A GAFF2"] --> A61["06A PHA in water (GROMACS)"] --> RC["GROMACS"]
    A5 --> A62["06_optional enzyme + polymer in water"] --> RA["OpenMM on GPU"]
    C5["05B CGenFF"] --> C62["06B PHA + enzyme in water"] --> RC
```

Use this notebook after the solvated system for your route exists:

- **GAFF2 PHA in water, GROMACS:** `gromacs/solvated_polymer/` from `06A_gaff2_gromacs_pha_in_water.ipynb` (the polymer benchmark method).
- **Amber/OpenMM:** the `OUTPUT_DIR` from `06_optional_gaff2_openmm_pha_enzyme_in_water.ipynb` (enzyme + polymer), with `run_openmm_md.py`, `protocol.json`, `run_step6_local.sh` and `run_hpc_equilibration_production.slurm`.
- **CHARMM/GROMACS, enzyme + polymer in water:** the `RUN_DIR` from `06B_cgenff_gromacs_pha_enzyme_in_water.ipynb`, with CHARMM-GUI's topology.

Both GROMACS folders use the `step6.x`/`step7` file names and the same two scripts.

Older GAFF2 → GROMACS folders (`gromacs/solvated_polymer/`, as used for the polymer-only benchmark) also work with the GROMACS cells below.

In [1]:
from pathlib import Path
import sys

repo_root = Path.cwd() if (Path.cwd() / "src" / "iphasimulator").exists() else Path.cwd().parent
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

output_root = repo_root / "examples" / "output"
from iphasimulator.naming import oligomer_name

system_name = oligomer_name("3HB", 4)
md_root = output_root / "md_tests" / system_name
print(f"Repository: {repo_root}")
print(f"System: {system_name}")
print(f"MD output root: {md_root}")

gromacs_solvated_dir = md_root / "gromacs" / "solvated_polymer"
openmm_dry_dir = md_root / "openmm" / "dry_polymer"
openmm_solvated_dir = md_root / "openmm" / "solvated_polymer"

{
    "gromacs_solvated_hpc_script": gromacs_solvated_dir / "run_hpc_equilibration_production.slurm",
    "gromacs_script_exists": (gromacs_solvated_dir / "run_hpc_equilibration_production.slurm").exists(),
    "openmm_dry_dir": openmm_dry_dir,
    "openmm_solvated_dir": openmm_solvated_dir,
}

Repository: /Users/k20098771/opt/iPHASimulator_v2
System: P3HB_4
MD output root: /Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/P3HB_4


{'gromacs_solvated_hpc_script': PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/P3HB_4/gromacs/solvated_polymer/run_hpc_equilibration_production.slurm'),
 'gromacs_script_exists': True,
 'openmm_dry_dir': PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/P3HB_4/openmm/dry_polymer'),
 'openmm_solvated_dir': PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/P3HB_4/openmm/solvated_polymer')}

## Local vs HPC Execution

Local runs are for smoke testing a few hundred to a few thousand steps. HPC runs are for longer NVT/NPT/production stages, restart continuation, and benchmarking.

Keep generated trajectories, checkpoints, and logs under `examples/output/` or scratch storage. They are ignored by Git by default.


## OpenMM on GPU (Amber force fields, from 06_optional)

The OpenMM folder uses the same stage names as the GROMACS folders. `run_step6_local.sh` runs step 6.0 (minimisation) locally; `run_hpc_equilibration_production.slurm` runs `python run_openmm_md.py step6.1_nvt step6.2_npt step7_production` on one GPU. `run_openmm_md.py` is stand-alone (it needs only OpenMM), reads its settings from `protocol.json` (the 06_optional enzyme protocol, matching 06B), and continues production from `step7_production.chk` when the job is resubmitted. Edit the environment-activation line in the SLURM file for your cluster.

In [ ]:
OPENMM_RUN_DIR = None  # the OUTPUT_DIR used in 06_optional

if OPENMM_RUN_DIR is None:
    print("Set OPENMM_RUN_DIR to the 06_optional OUTPUT_DIR to check the OpenMM files.")
else:
    openmm_run_dir = Path(OPENMM_RUN_DIR)
    for name in ("system.prmtop", "system.inpcrd", "run_openmm_md.py", "protocol.json",
                 "step6.0_minimization.xml", "run_hpc_equilibration_production.slurm"):
        print(f"{'found  ' if (openmm_run_dir / name).exists() else 'missing'}  {name}")
    print("Run step 6.0 locally first (bash run_step6_local.sh) if step6.0_minimization.xml is missing.")
    print("Submit from a terminal on the cluster with:")
    print(f"cd {openmm_run_dir}")
    print("sbatch run_hpc_equilibration_production.slurm")

### OpenMM GPU settings

For OpenMM, choose the CUDA platform in the workflow script or notebook settings after validating CPU execution:

```python
platform_name = "CUDA"
platform_precision = "mixed"
```

Use checkpoints and serialized states (`state.xml`) for restart continuation. `run_openmm_md.py` writes `step7_production.chk` and restarts production from it automatically when resubmitted. Keep restart files in the workflow output folder.


## GROMACS (step6.x/step7, from 06A or 06B)

A GROMACS run folder contains `step5_input.gro`, `topol.top` with its include files (the water/ion files for 06A; `toppar/` for 06B), `index.ndx`, `step6.0_minimization.mdp`, `step6.1_nvt.mdp`, `step6.2_npt.mdp`, `step7_production.mdp`, `run_step6_local.sh` and `run_hpc_equilibration_production.slurm`. Minimise locally (step 6.0), then submit steps 6.1, 6.2 and 7 with SLURM.

Set `GROMACS_RUN_DIR` to the `RUN_DIR` from 06B. Left as `None`, the cells below use `gromacs/solvated_polymer/` from the setup cell, which is where 06A writes its system.

In [ ]:
GROMACS_RUN_DIR = None  # 06A: None (setup-cell folder); 06B: RUN_DIR

if GROMACS_RUN_DIR is not None:
    gromacs_solvated_dir = Path(GROMACS_RUN_DIR)
    print(f"CHARMM/GROMACS run folder: {gromacs_solvated_dir}")
else:
    print(f"GROMACS_RUN_DIR is None; using the example folder from the setup cell: {gromacs_solvated_dir}")

### GROMACS SLURM submission

Submit only after 06B has prepared the run folder and all its checks pass. Neither 06B nor the setup cells run minimisation by default: run step 6.0 in 06B or with the optional local minimisation cell below (`RUN_LOCAL_MINIMIZATION = True`).

Then confirm `step6.0_minimization.gro` exists before submitting the SLURM job.

This HPC script has been tested on the KCL GPU cluster using `gromacs/2021.5-gcc-11.4.0-cuda-11.8.0`. Users on other HPC systems may need to modify the SLURM directives and module load command.


In [2]:
import subprocess

RUN_LOCAL_MINIMIZATION = False

local_minimization_script = gromacs_solvated_dir / "run_step6_local.sh"
minimized_structure = gromacs_solvated_dir / "step6.0_minimization.gro"

if RUN_LOCAL_MINIMIZATION:
    result = subprocess.run(
        ["bash", local_minimization_script.name],
        cwd=gromacs_solvated_dir,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()
else:
    print("Set RUN_LOCAL_MINIMIZATION = True to run GROMACS minimization in this notebook.")

print(f"Local minimization script: {local_minimization_script}")
print(f"Minimized structure: {minimized_structure}")
print(f"Minimized structure exists: {minimized_structure.exists()}")

Set RUN_LOCAL_MINIMIZATION = True to run GROMACS minimization in this notebook.
Local minimization script: /Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/P3HB_4/gromacs/solvated_polymer/run_step6_local.sh
Minimized structure: /Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/P3HB_4/gromacs/solvated_polymer/step6.0_minimization.gro
Minimized structure exists: False


In [3]:
gromacs_slurm_script = gromacs_solvated_dir / "run_hpc_equilibration_production.slurm"
local_minimization_script = gromacs_solvated_dir / "run_step6_local.sh"
minimized_structure = gromacs_solvated_dir / "step6.0_minimization.gro"

print(f"Script: {gromacs_slurm_script}")
print(f"Exists: {gromacs_slurm_script.exists()}")
print(f"Minimized structure: {minimized_structure}")
print(f"Minimized structure exists: {minimized_structure.exists()}")
if not minimized_structure.exists():
    print("Run the optional local minimization cell above first by setting RUN_LOCAL_MINIMIZATION = True.")
    print("Then submit from a terminal on the cluster with:")
else:
    print("Submit from a terminal on the cluster with:")
print(f"cd {gromacs_solvated_dir}")
print("sbatch run_hpc_equilibration_production.slurm")

Script: /Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/P3HB_4/gromacs/solvated_polymer/run_hpc_equilibration_production.slurm
Exists: True
Minimized structure: /Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/P3HB_4/gromacs/solvated_polymer/step6.0_minimization.gro
Minimized structure exists: False
Run the optional local minimization cell above first by setting RUN_LOCAL_MINIMIZATION = True.
Then submit from a terminal on the cluster with:
cd /Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/P3HB_4/gromacs/solvated_polymer
sbatch run_hpc_equilibration_production.slurm


### `ntmpi` and `ntomp`

For a single GPU GROMACS job, a common starting point is one MPI rank and several OpenMP threads:

```bash
gmx mdrun -deffnm step7_production -ntmpi 1 -ntomp 8 -pin on
```

Tune `-ntomp` to the CPU cores allocated per GPU. Too many threads can reduce performance. Multi-GPU jobs require deliberate domain decomposition and benchmarking.


## Restart and Continuation Checklist

- keep `state.xml`, checkpoint files, final coordinates, topology, and MDP/config files together
- record the exact executable/module/conda environment used
- restart GROMACS from the latest `.cpt` with `gmx mdrun -cpi`
- restart OpenMM from `checkpoint.chk` or `state.xml`
- benchmark short segments before launching long production jobs
- compare ns/day after changing GPU, `ntomp`, precision, PME settings, or report intervals


## Config-Driven Batch Workflow

The repository also includes a config-driven validation runner for scripted local or SLURM use. It writes OpenMM dry outputs to `openmm/dry_polymer/` and keeps generated files under ignored output folders.


In [4]:
from iphasimulator.workflows import load_workflow_config, workflow_plan, render_slurm_script

config_path = repo_root / "examples" / "hpc_validation_workflow.yaml"
config = load_workflow_config(config_path)
for item in workflow_plan(config):
    print(item)

build P3HB_4 -> examples/output/polymer_structures/P3HB_4.sdf
gaff2 P3HB_4 -> examples/output/md_tests/P3HB_4/gaff2
openmm dry P3HB_4 -> examples/output/md_tests/P3HB_4/openmm/dry_polymer
build P3HO_4 -> examples/output/polymer_structures/P3HO_4.sdf
gaff2 P3HO_4 -> examples/output/md_tests/P3HO_4/gaff2
openmm dry P3HO_4 -> examples/output/md_tests/P3HO_4/openmm/dry_polymer
build P3HDD_4 -> examples/output/polymer_structures/P3HDD_4.sdf
gaff2 P3HDD_4 -> examples/output/md_tests/P3HDD_4/gaff2
openmm dry P3HDD_4 -> examples/output/md_tests/P3HDD_4/openmm/dry_polymer


In [5]:
script_text = render_slurm_script(
    config_path="examples/hpc_validation_workflow.yaml",
    repo_root=repo_root,
    config=config,
)
print(script_text)

#!/usr/bin/env bash
#SBATCH --job-name=ipha_validation
#SBATCH --time=02:00:00
#SBATCH --partition=standard
#SBATCH --cpus-per-task=4
#SBATCH --mem=8G
#SBATCH --output=logs/%x-%j.out
#SBATCH --error=logs/%x-%j.err

set -euo pipefail

cd /Users/k20098771/opt/iPHASimulator_v2
mkdir -p logs

# Adjust this block to match your HPC module/conda setup.
source "$(conda info --base)/etc/profile.d/conda.sh"
conda activate ipha

PYTHONPATH=src python examples/run_configured_workflow.py --config examples/hpc_validation_workflow.yaml

